# 02 · Count audit and gate G1 — Neonatal mortality NCHS (M4)

| Field | Content |
|---|---|
| Framework sequence | 01: S1 · S2 · S3 → **02: S4 · G1** → 03: S5 · S6 · S7 · G2 · G3 → 04: S8 → 05: S8 · S9 · G4 → 06: S10 · S11 |
| What it does | Counts population and events by label and cohort, checks the coding of age at death, the frequencies and unknowns of every candidate predictor in the whole population (never split by outcome), and checks that the reference counts are reproduced |
| Stages and gates | S4 · G1 · preliminary G2 check (unknowns) |
| Decisions it led to | M4-D1 (48-hour landmark), M4-D2 (no marital status), M4-D3 (imputation without missingness indicators); see the outcome card |
| Inputs | `data/interim/cohort_<year>.parquet` (notebook 01) |
| Outputs | `data/outputs_02_count_audit.txt` (aggregate counts only; 1 to 9 shown as "<10") |
| Time and memory | A few minutes; reads one field at a time (≈1 GB) |
| Previous / next | `01_build_cohorts` / `03_predictors_g2_g3` |

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.nchs_reader import COHORTS, OUT_DIR

# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "en"
import importlib
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)      # decision texts are shown in full
import warnings
# scikit-learn >= 1.8 announces that LogisticRegression(penalty=...) will be replaced by l1_ratio. The models keep
# penalty (the hyperparameters chosen in notebook 04) so that results stay exactly as reported; the notice is hidden.
warnings.filterwarnings("ignore", message=".*'penalty' was deprecated.*", category=FutureWarning)
lines_es = []
if LANG == "es":                       # progress messages (print) also in Spanish
    import builtins
    def print(*args, **kwargs):
        builtins.print(*[translate(a, "es") if isinstance(a, str) and not a.startswith("\n== ") else a for a in args], **kwargs)

def write_outputs(path, header):
    # English text output (cited by the documents) and, from the Spanish notebook, a Spanish copy (*_es.txt).
    path.write_text(header + "\n" + "\n".join(lines) + "\n", encoding="utf-8")
    if LANG == "es":
        path.with_name(path.stem + "_es.txt").write_text(translate(header, "es") + "\n" + "\n".join(lines_es) + "\n", encoding="utf-8")


OUT_FILE = ROOT / "data" / "outputs_02_count_audit.txt"
# Gate G1: reference counts of the data adapter (data/outputs_01_build_cohorts.txt):
# (population of 22-47 weeks, neonatal deaths 0-27 days)
EXPECTED = {2016: (3938288, 11497), 2017: (3848558, 11057), 2018: (3785243, 10703),
            2019: (3741187, 10249), 2020: (3607479, 9574), 2021: (3657762, 9633),
            2022: (3661524, 9967), 2023: (3589826, 9879)}
# "Unknown or not stated" codes (user guides). Character fields use "U"; DMAR is blank when the
# state does not report marital status. "Not applicable" codes (ILP_R 888, ME_TRIAL "X") are
# categories, not unknowns.
UNKNOWN = {"MRACEHISP": [8], "MEDUC": [9], "MBSTATE_REC": [3], "PAY_REC": [9], "BFACIL3": [9],
           "PRIORLIVE": [99], "PRIORDEAD": [99], "PRIORTERM": [99], "ILP_R": [999],
           "RF_CESARN": [99], "PRECARE": [99], "PREVIS": [99], "BMI": [99.9], "WTGAIN": [99],
           "ME_PRES": [9], "ME_ROUT": [9], "ATTEND": [9], "APGAR5": [99], "BRTHWGT": [9999]}
CHAR_UNKNOWN = "U"
BLOCKS = {
    "mother": ["MAGER", "MRACEHISP", "MEDUC", "DMAR", "MBSTATE_REC", "PAY_REC", "WIC"],
    "obstetric_history": ["PRIORLIVE", "PRIORDEAD", "PRIORTERM", "ILP_R", "RF_CESAR", "RF_CESARN", "RF_PPB"],
    "current_pregnancy": ["PRECARE", "PREVIS", "CIG_REC", "BMI", "WTGAIN", "RF_PDIAB", "RF_GDIAB",
                          "RF_PHYPE", "RF_GHYPE", "RF_EHYPE", "RF_INFT", "IP_GON", "IP_SYPH",
                          "IP_CHLAM", "IP_HEPB", "IP_HEPC"],
    "labor_delivery": ["LD_INDL", "LD_AUGM", "LD_STER", "LD_ANTB", "LD_CHOR", "LD_ANES", "ME_PRES",
                       "ME_ROUT", "ME_TRIAL", "BFACIL3", "ATTEND"],
    "newborn": ["DPLURAL", "SEX", "COMBGEST", "BRTHWGT", "APGAR5", "CA_ANEN", "CA_MNSB", "CA_CCHD",
                "CA_CDH", "CA_OMPH", "CA_GAST", "CA_LIMB", "CA_CLEFT", "CA_CLPAL", "CA_DOWN",
                "CA_DISOR", "CA_HYPO"],
}
FIELDS = [c for b in BLOCKS.values() for c in b]
NUMERIC = ["MAGER", "PRIORLIVE", "PRIORDEAD", "PRIORTERM", "ILP_R", "RF_CESARN", "PRECARE", "PREVIS",
           "BMI", "WTGAIN", "COMBGEST", "BRTHWGT", "APGAR5"]
CATEGORICAL = [c for c in FIELDS if c not in NUMERIC]
LABELS = ["label_main", "died_first_24h", "label_day1_27", "label_early_neonatal"]
DEATH = ["died_infant", "d_age_days", "d_ager5"]
lines = []

def show(title, obj):
    """Print and keep an aggregate table; counts from 1 to 9 are suppressed as '<10'."""
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        sup = lambda v: ("<10" if isinstance(v, (int, np.integer)) and not isinstance(v, (bool, np.bool_))
                         and 0 < v < 10 else v)
        txt = (obj.map(sup) if isinstance(obj, pd.DataFrame) else obj.map(sup)).to_string()
    else:
        txt = str(obj)
    block = f"\n== {title}\n{txt}"
    print(block)
    lines.append(block)

SUPPRESS_SMALL = True
import contextlib, io, html as _html
_show_en = show

def _fmt(v):
    # Display format of a table cell: thousands separator for counts, 4 significant digits for small values.
    if isinstance(v, (bool, np.bool_)) or v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    if SUPPRESS_SMALL and isinstance(v, (int, np.integer)) and 0 < v < 10:
        return "<10"
    if isinstance(v, (int, np.integer)) or (isinstance(v, (float, np.floating)) and float(v).is_integer() and abs(v) >= 1000):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        return f"{v:.4g}" if abs(v) < 1 else f"{v:,.4f}".rstrip("0").rstrip(".")
    return v

def _render(title, obj):
    # In Jupyter: a heading and an HTML table with scroll; elsewhere: the text block.
    try:
        from IPython import get_ipython
        from IPython.display import HTML, display as _display
        if get_ipython() is None:              # plain Python (not a notebook)
            raise ImportError
    except ImportError:
        return print(lines_es[-1] if LANG == "es" else lines[-1])
    head = f"<p style='margin:14px 0 4px 0'><b>{_html.escape(str(title))}</b></p>"
    if isinstance(obj, (pd.Series, pd.DataFrame)):
        if isinstance(obj, pd.Series):          # rows are different quantities: each cell on its own
            df = obj.astype(object).map(_fmt).to_frame(name=translate("value", LANG))
        else:                                    # columns: the same number of decimals down each column
            df = obj.copy().astype(object)
            for k in range(df.shape[1]):
                col = obj.iloc[:, k]
                if pd.api.types.is_float_dtype(col) and not col.dropna().map(float.is_integer).all():
                    dec = max((len(f"{v:.4g}".split(".")[1]) if "." in f"{v:.4g}" and "e" not in f"{v:.4g}" else 0)
                              for v in col.dropna()) if col.notna().any() else 0
                    df.iloc[:, k] = col.map(lambda v, d=dec: "" if pd.isna(v) else f"{v:,.{d}f}")
                else:
                    df.iloc[:, k] = col.map(_fmt)
        table = df.to_html(na_rep="", border=0)
        _display(HTML(head + f"<div style='max-height:420px; overflow:auto'>{table}</div>"))
    else:
        _display(HTML(head + f"<pre style='margin:0'>{_html.escape(str(obj))}</pre>"))

def show(title, obj):
    # The English block is kept in `lines` (data/outputs_*.txt) and, in Spanish, the translated block in
    # `lines_es` (data/outputs_*_es.txt); on screen the table is shown as an HTML table.
    with contextlib.redirect_stdout(io.StringIO()):
        _show_en(title, obj)
        if LANG == "es":
            _show_en(translate(title, "es"), translate(obj, "es"))
            lines_es.append(lines.pop())
    _render(translate(title, LANG), translate(obj, LANG))
def is_unknown(s, field):
    """True where the field is unknown or not stated (blank counts as unknown)."""
    if field == "BMI":                       # stored as float: compare with a tolerance
        return s.isna() | (s.astype("Float64") > 99.85).fillna(False)
    if field in UNKNOWN:
        return s.isna() | s.isin(UNKNOWN[field])
    if isinstance(s.dtype, pd.CategoricalDtype) or s.dtype == object:
        return s.isna() | (s.astype("string") == CHAR_UNKNOWN)
    return s.isna()

print("cohorts:", COHORTS)

## E4 · P1 · Count audit and definition gate — population, events by label and reproduction of the counts

Counts the population and deaths by label in each cohort and compares them with the reference counts of notebook 01.

| Result | What is done | In this model |
|---|---|---|
| Counts match in the 8 cohorts | G1 passes: the definition is fixed | Passes |
| Any cohort differs | It stops (`assert`): back to S2 or S3 until the definition is stable | — |

In [ ]:
# Q-N1 population and events by label; Q-N2 age at death in days; Q-N3 gestational-age bands;
# accumulators for Q-N4 to Q-N7 (whole population, never split by outcome)
pop_rows, age_rows, ga_rows, unk_year = [], [], [], []
cat_counts = {c: {} for c in CATEGORICAL}
num_counts = {c: {} for c in NUMERIC}
nunk_main, nunk_ge28 = {}, {}          # number of unknowns -> [records, neonatal deaths] (gate G2)
imputed = []
GA_BANDS = [(22, 23), (24, 27), (28, 31), (32, 36), (37, 47)]
for y in COHORTS:
    path = OUT_DIR / f"cohort_{y}.parquet"
    d = pd.read_parquet(path, columns=["RESTATUS", "COMBGEST", "COMBGST_IMP", "BWTIMP"] + LABELS + DEATH)
    keep = ((d["RESTATUS"] < 4) & d["COMBGEST"].between(22, 47)).to_numpy()
    d = d[keep]
    ge28 = d["COMBGEST"] >= 28
    pop_rows.append({"cohort": y, "population": len(d), "events_0_27d": int(d["label_main"].sum()),
                     "first_24h": int(d["died_first_24h"].sum()), "day1_27": int(d["label_day1_27"].sum()),
                     "early_0_6d": int(d["label_early_neonatal"].sum()),
                     "post_neonatal_28d_1y": int((d["died_infant"] & ~d["label_main"]).sum()),
                     "population_ge28": int(ge28.sum()), "events_ge28": int((ge28 & d["label_main"]).sum()),
                     "per_1000": round(1000 * d["label_main"].mean(), 2)})
    dead = d[d["label_main"]]
    a = dead["d_age_days"].astype("Int64")
    row = {"cohort": y, "under_1h": int((dead["d_ager5"] == 1).sum()), "1_23h": int((dead["d_ager5"] == 2).sum())}
    row.update({f"day_{k}": int((a == k).sum()) for k in range(1, 7)})
    row["day_7_27"] = int(a.between(7, 27).sum())
    row["age_missing"] = int(a.isna().sum())
    age_rows.append(row)
    for lo, hi in GA_BANDS:
        g = d["COMBGEST"].between(lo, hi)
        ga_rows.append({"cohort": y, "band": f"{lo}-{hi}", "population": int(g.sum()),
                        "events_0_27d": int((g & d["label_main"]).sum()),
                        "first_24h": int((g & d["died_first_24h"]).sum())})
    imputed.append({"cohort": y, "ga_imputed_pct": round(100 * (d["COMBGST_IMP"] == "1").mean(), 2),
                    "bw_imputed_pct": round(100 * (d["BWTIMP"] == "1").mean(), 2)})
    label = d["label_main"].to_numpy(bool)
    ge28 = ge28.to_numpy()
    del d, dead
    # one field at a time keeps the memory low (about 1 GB)
    n = np.zeros(int(keep.sum()), dtype=np.int16)
    row = {"cohort": y}
    for c in FIELDS:
        s = pd.read_parquet(path, columns=[c])[c][keep].reset_index(drop=True)
        u = is_unknown(s, c).to_numpy(bool)
        n += u
        row[c] = round(100 * u.mean(), 2)
        if c in CATEGORICAL:
            for k, v in s.astype("object").where(s.notna(), "missing").value_counts().items():
                cat_counts[c][k] = cat_counts[c].get(k, 0) + int(v)
        else:
            for k, v in s.astype("Float64").fillna(-1).value_counts().items():
                num_counts[c][round(float(k), 1)] = num_counts[c].get(round(float(k), 1), 0) + int(v)
        del s, u
    unk_year.append(row)
    for store, mask in ((nunk_main, np.ones(len(n), bool)), (nunk_ge28, ge28)):
        recs = np.bincount(n[mask]); evs = np.bincount(n[mask], weights=label[mask], minlength=len(recs))
        for k in np.nonzero(recs)[0]:
            r = store.setdefault(int(k), [0, 0])
            r[0] += int(recs[k]); r[1] += int(evs[k])
    del n, label, ge28, keep
    print(y, "done")

pop = pd.DataFrame(pop_rows).set_index("cohort")
show("Q-N1 Population (US residents, 22-47 weeks) and events by label", pop)
g1 = pd.Series({y: (int(pop.loc[y, "population"]), int(pop.loc[y, "events_0_27d"])) == EXPECTED[y]
                for y in COHORTS}, name="reproduced")
show("Gate G1: reference counts reproduced", g1)
assert g1.all(), "gate G1: the counts do not match the data adapter"

## E4 · Count audit — age at death and gestational-age bands

The landmark of sensitivity (a) depends on how age at death is coded on days 0 and 1; this section checks that it is stable across cohorts. The output ends with the table “S4 decision: landmark…”, which shows the result of the check and the path taken. It is followed by “S4 decision: gestational age”. The reading criterion (a relative change of more than 20% in the share of deaths on day 1 between the first and the last cohort) was added when the notebooks were reorganized, so that the decision shows in the output; the decision itself was taken before fitting (outcome card).

| Result | What is done | In this model |
|---|---|---|
| Day-1 deaths stable across cohorts | The 24-hour landmark is kept | — |
| They change across cohorts | The landmark moves to a definition not affected by the change, recorded before fitting | Day 1: 1,412 (2016) → ~430 (2019+): **M4-D1, 48 h** |

In [ ]:
# Q-N2 age at death of the neonatal deaths: hours on day 0, then days
age = pd.DataFrame(age_rows).set_index("cohort")
show("Q-N2 Neonatal deaths by age at death (day 0 split into <1 h and 1-23 h)", age)
share = pd.DataFrame({"day0_pct": (100 * (age["under_1h"] + age["1_23h"]) / pop["events_0_27d"]).round(1),
                      "day1_pct": (100 * age["day_1"] / pop["events_0_27d"]).round(1),
                      "day0_plus_day1_pct": (100 * (age["under_1h"] + age["1_23h"] + age["day_1"])
                                             / pop["events_0_27d"]).round(1),
                      "day2_27_pct": (100 * (pop["events_0_27d"] - age["under_1h"] - age["1_23h"] - age["day_1"])
                                      / pop["events_0_27d"]).round(1)})
show("Q-N2b Share of the neonatal deaths: day 0, day 1, days 0-1 together and days 2-27", share)

# Q-N3 gestational-age bands, all cohorts together
ga = pd.DataFrame(ga_rows).groupby("band", sort=False)[["population", "events_0_27d", "first_24h"]].sum()
ga["pct_population"] = (100 * ga["population"] / ga["population"].sum()).round(2)
ga["pct_events"] = (100 * ga["events_0_27d"] / ga["events_0_27d"].sum()).round(1)
ga["per_1000"] = (1000 * ga["events_0_27d"] / ga["population"]).round(1)
show("Q-N3 Gestational-age bands (combined gestation), cohorts 2016-2023", ga)
show("Q-N3b Gestational age and birthweight imputed by NCHS (% of the population)",
     pd.DataFrame(imputed).set_index("cohort"))

# Decision of this section (M4-D1): is the coding of deaths on day 0 and day 1 stable across cohorts?
# Reading criterion: the share of the neonatal deaths that falls on day 1 changes by more than 20% (relative)
# between the first and the last cohort. Shares are used, not counts, because neonatal deaths also fall over time.
pd.set_option("display.max_colwidth", None)
d1 = age["day_1"]
rel = lambda s: 100 * (s.iloc[-1] - s.iloc[0]) / s.iloc[0]
stable = abs(rel(share["day1_pct"])) <= 20
show("S4 decision: landmark of sensitivity analysis (a) (M4-D1)", pd.Series({
    "day-1 deaths, first vs last cohort": f"{int(d1.iloc[0])} vs {int(d1.iloc[-1])}",
    "% of the deaths on day 1, first vs last cohort": f"{share['day1_pct'].iloc[0]} vs {share['day1_pct'].iloc[-1]}",
    "relative change of that share (%)": round(rel(share["day1_pct"]), 1),
    "% of the deaths on days 0-1 together, first vs last cohort": f"{share['day0_plus_day1_pct'].iloc[0]} vs {share['day0_plus_day1_pct'].iloc[-1]}",
    "relative change of that share (%) ": round(rel(share["day0_plus_day1_pct"]), 1),
    "day-1 coding stable (change within 20%)": bool(stable),
    "decision": "24-hour landmark kept" if stable else
                "M4-D1: landmark moved to 48 hours (days 0-1 together do not depend on how day 0 and day 1 are split)"}))

# What the gestational-age bands lead to
ga_short = ga.loc[["22-23", "24-27"]]
imp = pd.DataFrame(imputed).set_index("cohort")
show("S4 decision: gestational age", pd.Series({
    "% of births at 22-27 weeks": round(ga_short["pct_population"].sum(), 2),
    "% of deaths at 22-27 weeks": round(100 * ga_short["events_0_27d"].sum() / ga["events_0_27d"].sum(), 1),
    "consequence": "sensitivity analysis (c) restricted to 28 weeks or more (outcome card)",
    "gestational age imputed by NCHS, highest cohort (%)": float(imp["ga_imputed_pct"].max()),
    "consequence for the imputed values": "kept as recorded; the imputation flags stay as metadata, not as predictors"}))

## E4 · Count audit — frequencies of the candidate predictors

Frequencies and percentiles of every candidate in the whole population, never split by outcome, and % unknown by cohort. The output ends with the table “S4 decision: candidates with more than 5% unknown (M4-D2)”, which shows the result of the check and the path taken.

| Result | What is done | In this model |
|---|---|---|
| Candidate with more than 5% unknown or a reporting change across cohorts | Decide before S7 whether it enters, with which handling, or is excluded | Marital status (0% in 2016, ~12% from 2017): **M4-D2, excluded**. Interpregnancy interval (11.6%): enters |
| Other candidates (≤ 3% unknown) | Enter S7 | Yes |

In [ ]:
# Q-N4 frequencies of the categorical candidates (whole population, never split by outcome)
total = int(pop["population"].sum())
for block, cols in BLOCKS.items():
    for c in cols:
        if c in CATEGORICAL:
            s = pd.Series(cat_counts[c])
            s = s.reindex(sorted(s.index, key=str))
            show(f"Q-N4 [{block}] {c}: births and % of the population",
                 pd.DataFrame({"n": s, "pct": (100 * s / total).round(2)}))

# Q-N5 numeric candidates: unknown and percentiles (unknown codes and the 888 'not applicable' excluded)
rows = []
for c in NUMERIC:
    s = pd.Series(num_counts[c])
    excl = set(UNKNOWN.get(c, [])) | {-1.0} | ({888.0} if c == "ILP_R" else set())
    unk = s[[k for k in s.index if k in excl and k != 888.0]].sum()
    v = s[[k for k in s.index if k not in excl and not (c == "BMI" and abs(k - 99.9) < 1e-3)]].sort_index()
    cum = v.cumsum() / v.sum()
    row = {"field": c, "pct_unknown": round(100 * unk / total, 2)}
    row.update({f"p{int(q * 100):02d}": float(v.index[np.searchsorted(cum.values, q)])
                for q in (0.05, 0.25, 0.5, 0.75, 0.95)})
    if c == "ILP_R":
        row["pct_not_applicable_888"] = round(100 * s.get(888.0, 0) / total, 2)
    rows.append(row)
show("Q-N5 Numeric candidates: % unknown and percentiles", pd.DataFrame(rows).set_index("field"))

# Q-N6 % unknown of every candidate by cohort (whole population)
uy = pd.DataFrame(unk_year).set_index("cohort").T
uy["all_cohorts"] = (uy.mul(pop["population"], axis=1).sum(axis=1) / total).round(2)
show("Q-N6 % unknown by cohort, whole population", uy)
show("Q-N6b Candidates with more than 5% unknown in any cohort", uy[uy.drop(columns="all_cohorts").max(axis=1) > 5])

# Decision of this section (M4-D2): candidates with more than 5% unknown in any cohort.
# A field reported by every state in some cohorts (under 1% unknown) and missing in others (over 5%) has a
# reporting change: its unknowns depend on the cohort, not on the birth.
by_year = uy.drop(columns="all_cohorts")
high = by_year[by_year.max(axis=1) > 5].index
dec = {}
for f in high:
    reporting_change = bool(by_year.loc[f].min() < 1)
    dec[f] = {"% unknown, all cohorts": float(uy.loc[f, "all_cohorts"]),
              "% unknown, lowest-highest cohort": f"{by_year.loc[f].min():.1f}-{by_year.loc[f].max():.1f}",
              "reporting change across cohorts": reporting_change,
              "decision": "M4-D2: excluded" if reporting_change else "enters S7 (unknowns imputed)"}
others = uy.drop(index=high)
dec["other candidates"] = {"% unknown, all cohorts": float(others["all_cohorts"].max()),
                           "% unknown, lowest-highest cohort": f"up to {others.drop(columns='all_cohorts').max().max():.1f}",
                           "reporting change across cohorts": False, "decision": "enter S7"}
show("S4 decision: candidates with more than 5% unknown (M4-D2)", pd.DataFrame(dec).T)

## P2 · Leakage gate (preliminary check) — does the number of unknowns alone separate the deaths?

If unknowns concentrate in the events, a model could learn how the certificate was filled in rather than the risk. The output ends with the table “Preliminary G2 decision: handling of unknowns (M4-D3)”, which shows the result of the check and the path taken. The reading criterion (AUROC above 0.55) was added when the notebooks were reorganized, so that the decision shows in the output; the decision itself was taken before fitting (outcome card).

| Result | What is done | In this model |
|---|---|---|
| AUROC of the number of unknowns ≈ 0.50 | No special handling | — |
| Clearly above 0.50 | Fixed before fitting: imputation without missingness indicators and a complete-record sensitivity analysis | 0.577 (0.556 at ≥ 28 weeks): **M4-D3** |

In [ ]:
# Q-N7 leakage check (gate G2): does the number of unknown fields, by itself, separate the deaths?
def auroc_from_counts(table):
    """AUROC of an integer score from a {score: [records, events]} table (ties count one half)."""
    t = pd.DataFrame(table, index=["records", "events"]).T.sort_index()
    ev = t["events"].to_numpy(float); ne = (t["records"] - t["events"]).to_numpy(float)
    below = np.concatenate([[0], np.cumsum(ne)[:-1]])
    return float((ev * (below + 0.5 * ne)).sum() / (ev.sum() * ne.sum()))

def summary(table):
    t = pd.DataFrame(table, index=["records", "events"]).T.sort_index()
    t.index = [str(k) if k < 5 else "5+" for k in t.index]
    return t.groupby(level=0, sort=False).sum()

g2 = pd.DataFrame({"population": [auroc_from_counts(nunk_main)], "ge28_weeks": [auroc_from_counts(nunk_ge28)]},
                  index=["AUROC of the number of unknowns"]).round(3)
show("Q-N7 Gate G2: AUROC of the number of unknown fields alone (neonatal death 0-27 days)", g2)
show("Q-N7b Number of unknown fields: births (whole population)", summary(nunk_main)[["records"]])

# Preliminary G2 decision (M4-D3): does the number of unknowns alone separate the deaths?
# Reading criterion: an AUROC above 0.55 counts as clearly above chance (0.50).
auc_pop, auc_28 = float(g2.loc["AUROC of the number of unknowns", "population"]), float(g2.loc["AUROC of the number of unknowns", "ge28_weeks"])
above = auc_pop > 0.55
show("Preliminary G2 decision: handling of unknowns (M4-D3)", pd.Series({
    "AUROC of the number of unknowns (population / 28 weeks or more)": f"{auc_pop:.3f} / {auc_28:.3f}",
    "clearly above 0.50 (AUROC > 0.55)": bool(above),
    "decision": "M4-D3: imputation without missingness indicators, plus a complete-record sensitivity analysis" if above
                else "no special handling of unknowns"}))

write_outputs(OUT_FILE, "Count audit of the neonatal population (aggregate counts only; 1-9 shown as <10)")
print("written:", OUT_FILE)

## Notebook summary

Main indicators of the audit.

In [ ]:
# Summary of notebook 02
ga_ = ga.copy()
summary = pd.Series({
    "population (US residents, 22-47 weeks)": int(pop["population"].sum()),
    "neonatal deaths 0-27 days": int(pop["events_0_27d"].sum()),
    "gate G1 (reference counts reproduced)": "passes" if bool(g1.all()) else "DOES NOT PASS",
    "% of deaths at 22-27 weeks": round(100 * ga_.loc[["22-23", "24-27"], "events_0_27d"].sum() / ga_["events_0_27d"].sum(), 1),
    "% of births at 22-27 weeks": round(ga_.loc[["22-23", "24-27"], "pct_population"].sum(), 2),
    "day-1 deaths, first vs last cohort": f"{int(age['day_1'].iloc[0])} vs {int(age['day_1'].iloc[-1])}",
    "candidates with more than 5% unknown in any cohort": ", ".join(uy[uy.drop(columns='all_cohorts').max(axis=1) > 5].index) or "none",
    "AUROC of the number of unknowns alone": float(g2.loc["AUROC of the number of unknowns", "population"])}, name="value")
translate(summary.to_frame(), LANG)